# Compute reduced density matrices

A reduced density matrix (RDM) describes the state of a subsystem of particles obtained by tracing out the remaining degrees of freedom. In fermionic quantum chemistry, the expectation value of any $k$-body operator depends solely on the $k$-particle RDM rather than the full state vector.

This guide shows how to compute 1-, 2-, 3-, and 4-RDMs from an `ffsim` state vector, how to calculate RDMs of Slater determinants directly, and how to use 1- and 2-RDMs to evaluate molecular Hamiltonian expectation values.

`ffsim` provides the following tools for working with RDMs:

- [ffsim.rdms](../api/stubs/ffsim.rdms.rst#ffsim.rdms): Computes 1- through 4-body RDMs of an arbitrary state vector.
- [ffsim.slater_determinant_rdms](../api/stubs/ffsim.slater_determinant_rdms.rst#ffsim.slater_determinant_rdms): Computes 1- and 2-RDMs of a Slater determinant directly without constructing the full state vector.
- [ffsim.ReducedDensityMatrix](../api/stubs/ffsim.ReducedDensityMatrix.rst#ffsim.ReducedDensityMatrix): A container class for one- and two-body RDMs with methods to evaluate molecular Hamiltonian expectation values and rotate orbital bases.

## Compute RDMs from a state vector

To compute RDMs from a state vector, use [ffsim.rdms](../api/stubs/ffsim.rdms.rst#ffsim.rdms). The function takes:

- `vec`: The state vector (a 1D NumPy array of dimension $\binom{\text{norb}}{N_\alpha} \binom{\text{norb}}{N_\beta}$).
- `norb`: The number of spatial orbitals.
- `nelec`: A tuple `(n_alpha, n_beta)` specifying the number of alpha and beta electrons.
- `rank`: The maximum rank of RDM to return (from 1 to 4).

When `rank=1`, `ffsim.rdms` returns the 1-RDM directly as a single NumPy array. When `rank > 1`, it returns a tuple containing all RDMs up to and including the requested rank in increasing order (e.g., `(rdm1, rdm2)` for `rank=2`).

The following example computes the 1- and 2-RDMs of a random state vector.

In [ ]:
import numpy as np

import ffsim

# Initialize pseudorandom number generator
rng = np.random.default_rng(12345)

# Define system: 4 spatial orbitals, 2 alpha and 2 beta electrons
norb = 4
nelec = (2, 2)

# Generate a random state vector
dim = ffsim.dim(norb, nelec)
vec = ffsim.random.random_state_vector(dim, seed=rng)

# Compute 1- and 2-RDMs (spin-resolved by default)
rdm1, rdm2 = ffsim.rdms(vec, norb=norb, nelec=nelec, rank=2)

print("1-RDM shape (spin-resolved):", rdm1.shape)
print("2-RDM shape (spin-resolved):", rdm2.shape)

## RDM shapes and spin blocks

By default, `spin_summed=False`, so `ffsim.rdms` returns spin-resolved tensors. For a rank-$k$ RDM, the tensor has shape `(k + 1,) + (norb,) * (2 * k)`.

The leading axis indexes spin blocks with all $\alpha$ pairs preceding all $\beta$ pairs:

- **Rank 1** (shape `(2, norb, norb)`): `[0]` is $\alpha\alpha$ and `[1]` is $\beta\beta$.
- **Rank 2** (shape `(3, norb, norb, norb, norb)`): `[0]` is $\alpha\alpha$, `[1]` is $\alpha\beta$, and `[2]` is $\beta\beta$.

For normalized state vectors, the trace of each spin block of the 1-RDM equals the number of electrons in that spin sector:

$$\operatorname{Tr}(\text{rdm1}_\alpha) = N_\alpha, \quad \operatorname{Tr}(\text{rdm1}_\beta) = N_\beta$$

Passing `spin_summed=True` sums over all spin assignments, returning tensors of shape `(norb,) * (2 * k)`:

- Spin-summed 1-RDM: $\text{rdm1}_{\alpha\alpha} + \text{rdm1}_{\beta\beta}$, with trace equal to $N_\alpha + N_\beta$.
- Spin-summed 2-RDM: sum over all spin blocks $(\alpha\alpha, \alpha\beta, \beta\alpha, \beta\beta)$.

In [ ]:
# Verify particle conservation via spin-resolved traces
n_alpha_measured = np.trace(rdm1[0]).real
n_beta_measured = np.trace(rdm1[1]).real

print(f"Alpha electron count: {n_alpha_measured:.6f}")
print(f"Beta electron count:  {n_beta_measured:.6f}")
assert np.isclose(n_alpha_measured, nelec[0])
assert np.isclose(n_beta_measured, nelec[1])

# Compute spin-summed RDMs
rdm1_summed, rdm2_summed = ffsim.rdms(
    vec, norb=norb, nelec=nelec, rank=2, spin_summed=True
)

print("1-RDM shape (spin-summed):", rdm1_summed.shape)
print("2-RDM shape (spin-summed):", rdm2_summed.shape)
print(f"Total electron count:     {np.trace(rdm1_summed).real:.6f}")
assert np.isclose(np.trace(rdm1_summed).real, sum(nelec))

## Index ordering and the `reorder` parameter

The `reorder` argument controls how creation and annihilation operators are ordered in the definition of the RDMs.

### `reorder = True` (default)

When `reorder=True`, all creation operators appear before all annihilation operators:

- Rank 1: $\text{rdm1}[p, q] = \langle a_p^\dagger a_q \rangle$
- Rank 2: $\text{rdm2}[p, q, r, s] = \langle a_p^\dagger a_r^\dagger a_s a_q \rangle$
- Rank 3: $\text{rdm3}[p, q, r, s, t, u] = \langle a_p^\dagger a_r^\dagger a_t^\dagger a_u a_s a_q \rangle$
- Rank 4: $\text{rdm4}[p, q, r, s, t, u, v, w] = \langle a_p^\dagger a_r^\dagger a_t^\dagger a_v^\dagger a_w a_u a_s a_q \rangle$

In `rdm2`, the creation operators are $p^\dagger, r^\dagger$ and the annihilation operators are $s, q$. This convention directly matches the standard definition of the two-body molecular Hamiltonian in quantum chemistry:

$$H_{\text{two}} = \frac{1}{2} \sum_{\substack{pqrs \\ \sigma \tau}} h_{pqrs} a_{p\sigma}^\dagger a_{r\tau}^\dagger a_{s\tau} a_{q\sigma}$$

Because the indices match, contracting $h_{pqrs}$ with `rdm2[p, q, r, s]` requires no index transposition.

### `reorder = False`

When `reorder=False`, creation and annihilation operators alternate in pairs:

- Rank 2: $\text{rdm2}[p, q, r, s] = \langle a_p^\dagger a_q a_r^\dagger a_s \rangle$
- Rank 3: $\text{rdm3}[p, q, r, s, t, u] = \langle a_p^\dagger a_q a_r^\dagger a_s a_t^\dagger a_u \rangle$
- Rank 4: $\text{rdm4}[p, q, r, s, t, u, v, w] = \langle a_p^\dagger a_q a_r^\dagger a_s a_t^\dagger a_u a_v^\dagger a_w \rangle$

Without reordering, tracing out the second pair $(r, s)$ with $r=s$ corresponds to the number operator $\sum_r a_r^\dagger a_r = N$, so $\sum_r \text{rdm2}[p, q, r, r] = N \cdot \text{rdm1}[p, q]$. With reordering, the traced operator acts after the remaining same-spin annihilator, reducing the prefactor by one to $(N - 1) \cdot \text{rdm1}[p, q]$.

In [ ]:
# Compute 2-RDM without reordering
rdm1_no_reorder, rdm2_no_reorder = ffsim.rdms(
    vec, norb=norb, nelec=nelec, rank=2, spin_summed=True, reorder=False
)

# The 1-RDM is identical under both conventions (<p+ q>)
np.testing.assert_allclose(rdm1_summed, rdm1_no_reorder)

# Tracing the second pair in the non-reordered 2-RDM yields N * rdm1
traced_no_reorder = np.trace(rdm2_no_reorder, axis1=2, axis2=3)
expected_no_reorder = sum(nelec) * rdm1_summed
np.testing.assert_allclose(traced_no_reorder, expected_no_reorder, atol=1e-12)

# Tracing the second pair in the reordered 2-RDM yields (N - 1) * rdm1
traced_reorder = np.trace(rdm2_summed, axis1=2, axis2=3)
expected_reorder = (sum(nelec) - 1) * rdm1_summed
np.testing.assert_allclose(traced_reorder, expected_reorder, atol=1e-12)

## Evaluate energy with `ReducedDensityMatrix`

Given the one- and two-body RDMs of a state, the expectation value of any [MolecularHamiltonian](../api/stubs/ffsim.MolecularHamiltonian.rst#ffsim.MolecularHamiltonian) can be computed using [ffsim.ReducedDensityMatrix](../api/stubs/ffsim.ReducedDensityMatrix.rst#ffsim.ReducedDensityMatrix).

The `ReducedDensityMatrix` dataclass stores the spin-summed 1- and 2-RDMs. Its `expectation` method computes:

$$E = h_0 + \sum_{pq} h_{pq} \Gamma_{pq} + \frac{1}{2} \sum_{pqrs} h_{pqrs} \Gamma_{pqrs}$$

Evaluating energy via RDMs avoids repeated matrix-vector multiplications on exponentially large state vectors, making it suitable for evaluating multiple Hamiltonians or optimizing orbital bases.

In [ ]:
# Create a random molecular Hamiltonian
mol_ham = ffsim.random.random_molecular_hamiltonian(norb, seed=rng)

# Wrap the spin-summed RDMs in a ReducedDensityMatrix object
rdm_obj = ffsim.ReducedDensityMatrix(one_rdm=rdm1_summed, two_rdm=rdm2_summed)

# Compute energy expectation value from RDMs
energy_rdm = rdm_obj.expectation(mol_ham)

# Compare with direct state vector expectation value using LinearOperator
linop = ffsim.linear_operator(mol_ham, norb=norb, nelec=nelec)
energy_vec = np.vdot(vec, linop @ vec).real

print(f"Energy from RDMs:         {energy_rdm:.8f}")
print(f"Energy from state vector: {energy_vec:.8f}")
np.testing.assert_allclose(energy_rdm, energy_vec, atol=1e-12)

## Slater determinant RDMs

For Slater determinants, [ffsim.slater_determinant_rdms](../api/stubs/ffsim.slater_determinant_rdms.rst#ffsim.slater_determinant_rdms) calculates 1- and 2-RDMs analytically using Wick's theorem without constructing the full state vector.

`slater_determinant_rdms` supports:

- Spinful systems: `occupied_orbitals = (occ_a, occ_b)`.
- Spinless systems: `occupied_orbitals = occ`.
- Orbital rotations: an optional unitary matrix (or pair of matrices for independent alpha/beta rotations).

The following example computes the RDMs of a Slater determinant directly from its occupied orbitals and orbital rotation matrix.

In [ ]:
# Define a Slater determinant: 10 orbitals, 5 alpha and 3 beta electrons
norb_slater = 10
n_alpha_slater, n_beta_slater = 5, 3
occupied_orbitals = (range(n_alpha_slater), range(n_beta_slater))

# Apply an orbital rotation
orbital_rotation = ffsim.random.random_unitary(norb_slater, seed=rng)

# Compute 1- and 2-RDMs directly
slater_rdm1, slater_rdm2 = ffsim.slater_determinant_rdms(
    norb_slater,
    occupied_orbitals,
    orbital_rotation=orbital_rotation,
    rank=2,
)

print("Slater 1-RDM shape:", slater_rdm1.shape)
print("Slater 2-RDM shape:", slater_rdm2.shape)
print(f"Alpha electron trace: {np.trace(slater_rdm1[0]).real:.1f}")
print(f"Beta electron trace:  {np.trace(slater_rdm1[1]).real:.1f}")
assert np.isclose(np.trace(slater_rdm1[0]).real, n_alpha_slater)
assert np.isclose(np.trace(slater_rdm1[1]).real, n_beta_slater)

## Higher-rank RDMs (3- and 4-RDMs)

`ffsim.rdms` supports computing 3- and 4-body reduced density matrices by setting `rank=3` or `rank=4`.

### Spin blocks

- **Rank 3** (shape `(4,) + (norb,) * 6`): The 4 spin blocks correspond to $(\alpha\alpha\alpha, \alpha\alpha\beta, \alpha\beta\beta, \beta\beta\beta)$.
- **Rank 4** (shape `(5,) + (norb,) * 8`): The 5 spin blocks correspond to $(\alpha\alpha\alpha\alpha, \alpha\alpha\alpha\beta, \alpha\alpha\beta\beta, \alpha\beta\beta\beta, \beta\beta\beta\beta)$.

### Memory and scaling considerations

A rank-$k$ spin-summed RDM stores $\text{norb}^{2k}$ complex numbers. For example, with $\text{norb}=10$, a 4-RDM has $10^8$ entries (approximately $1.6\text{ GB}$ for complex128).

Additionally, the underlying PySCF spin-free kernels used for spin-resolved 3- and 4-RDMs create intermediate real tensors of shape $(2 \cdot \text{norb})^{2k}$. Therefore, higher-rank RDMs should be computed for compact active spaces.

In [ ]:
# Small system for higher-rank RDMs: 3 orbitals, 2 alpha and 1 beta electron
norb_small = 3
nelec_small = (2, 1)
vec_small = ffsim.random.random_state_vector(
    ffsim.dim(norb_small, nelec_small), seed=rng
)

# Compute RDMs up to rank 3
rdm1_s, rdm2_s, rdm3_s = ffsim.rdms(
    vec_small, norb=norb_small, nelec=nelec_small, rank=3
)

print("3-RDM shape (spin-resolved):", rdm3_s.shape)
print(f"Number of spin blocks: {len(rdm3_s)} (aaa, aab, abb, bbb)")

# The system has 2 alpha and 1 beta electron:
# - aaa requires 3 alpha electrons -> vanishes identically
# - aab requires 2 alpha and 1 beta electrons -> non-zero
# - abb requires 2 beta electrons -> vanishes identically
# - bbb requires 3 beta electrons -> vanishes identically
print(f"Norm of aaa block (needs 3 alpha): {np.linalg.norm(rdm3_s[0]):.2e}")
print(f"Norm of aab block (needs 2 alpha, 1 beta): {np.linalg.norm(rdm3_s[1]):.6f}")
print(f"Norm of abb block (needs 2 beta):  {np.linalg.norm(rdm3_s[2]):.2e}")
print(f"Norm of bbb block (needs 3 beta):  {np.linalg.norm(rdm3_s[3]):.2e}")

assert np.allclose(rdm3_s[0], 0)
assert not np.isclose(np.linalg.norm(rdm3_s[1]), 0)
assert np.allclose(rdm3_s[2], 0)
assert np.allclose(rdm3_s[3], 0)

## Rotate RDMs

The `rotated` method of [ffsim.ReducedDensityMatrix](../api/stubs/ffsim.ReducedDensityMatrix.rst#ffsim.ReducedDensityMatrix) applies an orbital rotation $U$ to the reduced density matrices:

$$\Gamma_{pq}' = \sum_{ab} U_{pa}^* U_{qb} \Gamma_{ab}$$

$$\Gamma_{pqrs}' = \sum_{abcd} U_{pa}^* U_{qb} U_{rc}^* U_{sd} \Gamma_{abcd}$$

Rotating the RDMs by $U$ gives the same energy expectation value as evaluating the unrotated RDMs against the Hamiltonian rotated by $U^\dagger$:

$$\operatorname{rdm.rotated}(U)\text{.expectation}(H) = \operatorname{rdm.expectation}(H\text{.rotated}(U^\dagger))$$

Furthermore, rotating both the RDMs and the Hamiltonian by the same unitary leaves the energy invariant:

$$\operatorname{rdm.rotated}(U)\text{.expectation}(H\text{.rotated}(U)) = \operatorname{rdm.expectation}(H)$$

This transformation is used in orbital optimization algorithms such as [ffsim.optimize_orbitals](../api/stubs/ffsim.optimize_orbitals.rst#ffsim.optimize_orbitals), where the RDMs are kept fixed while the orbital rotation $U$ is optimized (see the [Optimize orbitals guide](optimize-orbitals.ipynb)).

In [ ]:
# Generate a random unitary orbital rotation
u = ffsim.random.random_unitary(norb, seed=rng)

# Rotate the RDMs
rdm_rotated = rdm_obj.rotated(u)

# Rotate the Hamiltonian by U^dagger
mol_ham_rotated = mol_ham.rotated(u.T.conj())

# Evaluating rotated RDMs against H equals unrotated RDMs against H(U^dagger)
energy_rdm_rotated = rdm_rotated.expectation(mol_ham)
energy_ham_rotated = rdm_obj.expectation(mol_ham_rotated)

print(f"Energy of rotated RDMs with H:         {energy_rdm_rotated:.8f}")
print(f"Energy of original RDMs with H(U^+):   {energy_ham_rotated:.8f}")
np.testing.assert_allclose(energy_rdm_rotated, energy_ham_rotated, atol=1e-12)

# Rotating both RDMs and Hamiltonian by U preserves the total energy
energy_both_rotated = rdm_rotated.expectation(mol_ham.rotated(u))
print(f"Energy with both rotated by U:         {energy_both_rotated:.8f}")
print(f"Original energy:                       {energy_rdm:.8f}")
np.testing.assert_allclose(energy_both_rotated, energy_rdm, atol=1e-12)